<a href="https://colab.research.google.com/github/saleemproven-jpg/AgenticAi/blob/main/Module_5_Tool_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Sprint 4 – Tool Calling & AI Agent Foundations**

**Client:** PROITBRIDGE Operations Manager

### **Business Requirement**
> Our AI Assistant can retrieve information from the Employee Handbook. It must now use external capabilities to perform tasks and access business systems.

This notebook introduces **Tool Calling → Multiple Tools → Agent Loop → Reasoning & Planning** without LangChain.


# **1. Install Required Libraries**

In [1]:
!pip -q install -U openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 892.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 16.9 MB/s eta 0:00:00


# **2. Configure OpenAI API**

In [2]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GENERATION_MODEL = "gpt-4.1-mini"

print("OpenAI client configured successfully.")
print("Model:", GENERATION_MODEL)


Enter your OpenAI API key: ··········
OpenAI client configured successfully.
Model: gpt-4.1-mini


# **3. First Tool — A Python Function**

A tool starts with a function that performs a specific task.


In [11]:
def calculate_leave_balance(total_leaves, leaves_taken):
    return {"remaining_leaves": total_leaves - leaves_taken}

result = calculate_leave_balance(24, 7)
print(result)

{'remaining_leaves': 17}


# **4. Function Calling - IMPORTANT NOTES**

1. The function exists in our application, but the **LLM does not execute Python code**.

2. The model receives a description of the available function and can return a **tool call** with the required arguments.

3. The application then executes the function and sends the result back to the model.


## **4.1 Define the Tool**

In [7]:
tools = [
    {
        "type": "function",
        "name": "calculate_leave_balance",
        "description": "Calculate an employee's remaining leave balance.",
        "parameters": {
            "type": "object",
            "properties": {
                "total_leaves": {
                    "type": "integer",
                    "description": "Total annual leave allowance."
                },
                "leaves_taken": {
                    "type": "integer",
                    "description": "Number of leaves already taken."
                }
            },
            "required": ["total_leaves", "leaves_taken"],
            "additionalProperties": False
        }
    }
]

## **4.2 Send the User Request to the Model**

In [8]:
question = "An employee has 24 total leaves and has taken 7. What is the remaining leave balance?"

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question,
                                    tools=tools)
response.output

[ResponseFunctionToolCall(arguments='{"total_leaves":24,"leaves_taken":7}', call_id='call_u18BXwYFHGU4iDZ8XzWoM084', name='calculate_leave_balance', type='function_call', id='fc_0597810c81602ecb006ac5a06c8a0487d09cf493bd5d5bc08d', async_=None, caller=None, namespace=None, status='completed')]

## **4.3 Execute the Requested Tool**

In [12]:
import json

for tool_call in response.output:
    arguments = json.loads(tool_call.arguments)
    result = calculate_leave_balance(**arguments)

    print("==="*30)
    print("Tool:", tool_call.name)
    print("Arguments:", arguments)
    print("Result:", result)

Tool: calculate_leave_balance
Arguments: {'total_leaves': 24, 'leaves_taken': 7}
Result: {'remaining_leaves': 17}


In [21]:
# here in this example i have only one tool / function so here i just given one name 4 times as duplicate
tool_functions = {
    "calculate_leave_balance": calculate_leave_balance,
    "get_employee_details": calculate_leave_balance,
    "check_attendance": calculate_leave_balance,
    "submit_leave": calculate_leave_balance
}

# **5. Complete Tool Calling Workflow**


In [22]:
input_items = [
    item.model_dump(by_alias=True, exclude_none=True)
    for item in response.output
]

for tool_call in response.output:
    arguments = json.loads(tool_call.arguments)
    function = tool_functions[tool_call.name]

    result = function(**arguments)

    input_items.append({"type": "function_call_output","call_id": tool_call.call_id,"output": json.dumps(result)  })
print(input_items)

final_response = client.responses.create(model=GENERATION_MODEL,input=input_items,tools=tools)

final_response.output_text

[{'arguments': '{"total_leaves":24,"leaves_taken":7}', 'call_id': 'call_u18BXwYFHGU4iDZ8XzWoM084', 'name': 'calculate_leave_balance', 'type': 'function_call', 'id': 'fc_0597810c81602ecb006ac5a06c8a0487d09cf493bd5d5bc08d', 'status': 'completed'}, {'type': 'function_call_output', 'call_id': 'call_u18BXwYFHGU4iDZ8XzWoM084', 'output': '{"remaining_leaves": 17}'}]


'You have 17 days of leave remaining for this year. If you would like, I can help you plan your leave days or provide information on leave policies.'

### ======================================================